# B3 model-building sandbox

Quick sanity checks for the `training.classification` code (feature loading, the
Configuration 3 chip join, and the RF/XGBoost sweep) against a small slice of the
real corpus, before running the full B3 sweep.

Requires the B2 patch/chip GeoParquet tables to already exist in S3 (B2.6) and AWS
credentials for `DEFAULT_BUCKET` (or pass a `--profile`-equivalent via
`boto3.Session(profile_name=...)` below).

In [ ]:
import boto3

from training.classification import (
    ALL_CONFIGURATIONS,
    configure_mlflow,
    load_feature_matrix,
    run_sweep,
)
from training.s3_paths import DEFAULT_BUCKET

BUCKET = DEFAULT_BUCKET
MAX_SCENES = 2  # small cap so this notebook runs in seconds, not the full 513/20 corpus

session = boto3.Session(profile_name="spk_data")
s3 = session.client("s3")

## 1. Sample each feature configuration

Loads the train split for a couple of scenes under each configuration and checks
the resulting matrix width against the feature contract (27 / 1046 / 2070 dims) --
this is the fast way to catch a schema drift or a broken chip join before spending
time on a full-corpus run.

In [ ]:
for configuration in ALL_CONFIGURATIONS:
    train = load_feature_matrix(
        s3, BUCKET, configuration, "train", is_pure_only=True, max_scenes=MAX_SCENES
    )
    print(f"{configuration.name}: X={train.X.shape}, y={train.y.shape}")
    print("  first 5 feature names:", train.feature_names[:5])

## 2. Capped sweep -- sanity-check before the full run

Runs all six (configuration x classifier) combinations on `MAX_SCENES` scenes and
logs each to a local MLflow file store (`./mlruns`), so this is safe to run
repeatedly without touching the SageMaker managed tracking server. Swap in the real
tracking URI (`--mlflow-tracking-uri` in `train_main.py`) once this looks right and
you're ready to run against the full corpus.

In [ ]:
configure_mlflow(tracking_uri="file:./mlruns", experiment_name="b3-notebook-sanity-check")

results = run_sweep(s3, BUCKET, is_pure_only=True, max_scenes=MAX_SCENES)

for result in sorted(results, key=lambda r: r.val_sic_r2, reverse=True):
    print(
        f"{result.configuration:8s} {result.classifier:14s} "
        f"val_sic_r2={result.val_sic_r2:.4f} val_ordinal_penalty={result.val_ordinal_penalty:.4f} "
        f"run_id={result.run_id}"
    )

## Next steps

Once these look right:

- Run the full sweep: `uv run python -m training.train_main --mlflow-tracking-uri <sagemaker-uri>`
  (drop `--max-scenes` to process the full corpus).
- After reviewing the sweep results and documenting the model-selection decision on
  the B3 issue, score the winner once on the test split and register it:
  `uv run python -m training.select_main --run-id <id> --configuration <name>`.